# 05. Monte Carlo Tournament Simulation
10,000 bracket simulations for Wimbledon 2026.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('imports ok')

In [ ]:
from src.simulation.draw_loader import load_wimbledon_2026_draw, get_bracket_summary

bracket = load_wimbledon_2026_draw()
summary = get_bracket_summary(bracket)

print('Bracket summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')

print(f'\nTotal matches in bracket: {len(bracket.matches)}')
print('Players in draw:')
seeded = [(m.player_a, m.player_b) for m in list(bracket.matches.values())[:5]]
for pa, pb in seeded:
    a_str = f'{pa.name} (S{pa.seed})' if pa and pa.seed else (pa.name if pa else 'TBD')
    b_str = f'{pb.name} (S{pb.seed})' if pb and pb.seed else (pb.name if pb else 'TBD')
    print(f'  {a_str} vs {b_str}')

In [ ]:
# Build WElo lookup from match history.
# Manually accumulate per-surface Elo ratings from scratch —
# mirrors the logic in src/features/elo.py::compute_elo_ratings.

fm = pd.read_parquet('../data/processed/feature_matrix.parquet')
matches = pd.read_parquet('../data/processed/matches_clean.parquet')

ELO_INITIAL = 1500.0
WELO_WEIGHTS = {'grass': 0.40, 'overall': 0.35, 'hard': 0.25}
K = 32.0

overall_elo: dict[str, float] = {}
grass_elo: dict[str, float] = {}
hard_elo: dict[str, float] = {}

df_sorted = matches.sort_values('tourney_date').reset_index(drop=True)

for _, row in df_sorted.iterrows():
    w, l = str(row['winner_id']), str(row['loser_id'])
    surface = str(row.get('surface', ''))

    ew = overall_elo.get(w, ELO_INITIAL)
    el = overall_elo.get(l, ELO_INITIAL)
    expected_w = 1.0 / (1.0 + 10 ** ((el - ew) / 400.0))
    overall_elo[w] = ew + K * (1 - expected_w)
    overall_elo[l] = el + K * (0 - (1 - expected_w))

    if surface == 'Grass':
        ew_g = grass_elo.get(w, ELO_INITIAL)
        el_g = grass_elo.get(l, ELO_INITIAL)
        exp_g = 1.0 / (1.0 + 10 ** ((el_g - ew_g) / 400.0))
        grass_elo[w] = ew_g + K * (1 - exp_g)
        grass_elo[l] = el_g + K * (0 - (1 - exp_g))

    if surface == 'Hard':
        ew_h = hard_elo.get(w, ELO_INITIAL)
        el_h = hard_elo.get(l, ELO_INITIAL)
        exp_h = 1.0 / (1.0 + 10 ** ((el_h - ew_h) / 400.0))
        hard_elo[w] = ew_h + K * (1 - exp_h)
        hard_elo[l] = el_h + K * (0 - (1 - exp_h))


def get_welo(player_id: str) -> float:
    """Grass-weighted Elo rating for a player."""
    overall = overall_elo.get(player_id, ELO_INITIAL)
    grass = grass_elo.get(player_id, ELO_INITIAL)
    hard = hard_elo.get(player_id, ELO_INITIAL)
    return (
        WELO_WEIGHTS['grass'] * grass
        + WELO_WEIGHTS['overall'] * overall
        + WELO_WEIGHTS['hard'] * hard
    )


print(f'Players with ratings: {len(overall_elo):,}')
sample_ids = list(overall_elo.keys())[:5]
for pid in sample_ids:
    print(f'  {pid}: welo={get_welo(pid):.1f}')

## Simulation

In [ ]:
from src.simulation.monte_carlo import simulate_tournament

N_SIMS = 10_000
SCALE = 400.0

def predict_fn(player_a_id: str, player_b_id: str) -> float:
    """P(A wins) using weighted Elo ratings."""
    welo_a = get_welo(player_a_id)
    welo_b = get_welo(player_b_id)
    return 1.0 / (1.0 + 10 ** ((welo_b - welo_a) / SCALE))

print(f'Running {N_SIMS:,} simulations...')
result = simulate_tournament(bracket, predict_fn, n_sims=N_SIMS, seed=42)
print('Done.')

print(f'\nTop 15 title probabilities:')
title_series = pd.Series(result.title_probs).head(15)
for name, prob in title_series.items():
    print(f'  {name:<30} {prob:.2%}')

In [ ]:
top15 = pd.Series(result.title_probs).head(15).sort_values()

colors = ['#2ecc71' if prob >= 0.10 else '#3498db' if prob >= 0.05 else '#95a5a6'
          for prob in top15.values]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top15.index, top15.values, color=colors, edgecolor='white')

for bar, prob in zip(bars, top15.values):
    ax.text(
        bar.get_width() + 0.002,
        bar.get_y() + bar.get_height() / 2,
        f'{prob:.1%}',
        va='center',
        fontsize=9
    )

ax.set_xlabel('Title probability')
ax.set_title(f'Wimbledon 2026 — Title Probabilities\n({N_SIMS:,} Monte Carlo simulations, Weighted Elo)')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
ax.set_xlim(0, top15.max() * 1.18)
plt.tight_layout()
plt.show()

print(f'\nTop 5 most likely finals:')
for pair, prob in list(result.final_matchups.items())[:5]:
    print(f'  {pair[0]} vs {pair[1]}: {prob:.1%}')